# 🔢 Handwritten Digit Recognition — Training Notebook

This notebook trains a simple CNN on the MNIST dataset to recognize handwritten digits (0–9).  
It is designed as **Stage 1** of a future handwritten mathematical expression recognition project.

**What this notebook covers:**
1. Loading the MNIST dataset
2. Preprocessing (grayscale normalization, reshaping)
3. Building a CNN model from scratch (no pretrained models)
4. Training & evaluating the model
5. Saving the trained model for later prediction

## 1. Install & Import Dependencies

In [ ]:
# Install dependencies (Colab already has TensorFlow, but this ensures versions)
!pip install tensorflow matplotlib numpy pillow scikit-learn -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import os

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 2. Load the MNIST Dataset

MNIST contains 70,000 grayscale images of handwritten digits (28×28 pixels):  
- **60,000** training images  
- **10,000** test images  

Each image is labeled with the digit it represents (0–9).

In [ ]:
# Load MNIST from Keras datasets
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print(f"Training set:  {X_train.shape}, labels: {y_train.shape}")
print(f"Test set:      {X_test.shape}, labels: {y_test.shape}")
print(f"Pixel range:   [{X_train.min()}, {X_train.max()}]")
print(f"Unique labels: {np.unique(y_train)}")

In [ ]:
# Visualize some sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("Sample MNIST Digits", fontsize=16, fontweight="bold")

for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap="gray")
    ax.set_title(f"Label: {y_train[i]}", fontsize=12)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 3. Preprocess the Data

We apply three preprocessing steps:

1. **Grayscale** — MNIST is already grayscale (single channel), so we add a channel dimension.
2. **Normalization** — Scale pixel values from [0, 255] → [0.0, 1.0] for faster and more stable training.
3. **Reshaping** — Reshape from (28, 28) → (28, 28, 1) so the CNN receives the expected 3D input.

Labels are one-hot encoded (e.g., digit `3` → `[0,0,0,1,0,0,0,0,0,0]`).

In [ ]:
# --- Preprocessing ---

# Target image dimensions
IMG_HEIGHT = 28
IMG_WIDTH = 28
NUM_CLASSES = 10

# 1. Normalize pixel values to [0, 1]
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# 2. Reshape to add channel dimension: (samples, 28, 28) -> (samples, 28, 28, 1)
X_train = X_train.reshape(-1, IMG_HEIGHT, IMG_WIDTH, 1)
X_test = X_test.reshape(-1, IMG_HEIGHT, IMG_WIDTH, 1)

# 3. One-hot encode labels
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_test_cat = to_categorical(y_test, NUM_CLASSES)

print(f"Preprocessed training data shape: {X_train.shape}")
print(f"Preprocessed test data shape:     {X_test.shape}")
print(f"Pixel range after normalization:  [{X_train.min()}, {X_train.max()}]")
print(f"One-hot label example (digit {y_train[0]}): {y_train_cat[0]}")

## 4. Build the CNN Model

We build a simple CNN from scratch — **no pretrained models**.

### Architecture Overview

```
Input (28×28×1)
  │
  ├─ Conv2D (32 filters, 3×3) + ReLU
  ├─ MaxPooling2D (2×2)
  │
  ├─ Conv2D (64 filters, 3×3) + ReLU
  ├─ MaxPooling2D (2×2)
  │
  ├─ Flatten
  ├─ Dense (128) + ReLU + Dropout (0.5)
  └─ Dense (10) + Softmax → Output (digit 0–9)
```

This is intentionally simple for Stage 1. Future stages will extend this for mathematical symbol recognition.

In [ ]:
def build_digit_cnn(input_shape=(28, 28, 1), num_classes=10):
    """
    Build a simple CNN for handwritten digit recognition.

    Architecture:
        - 2 convolutional blocks (Conv2D + MaxPool)
        - 1 fully connected layer with dropout
        - Softmax output for 10 digit classes

    Args:
        input_shape: Shape of input images (height, width, channels).
        num_classes: Number of output classes (default: 10 for digits 0-9).

    Returns:
        A compiled Keras Sequential model.
    """
    model = models.Sequential([
        # --- Block 1: First convolutional layer ---
        layers.Conv2D(32, (3, 3), activation="relu", input_shape=input_shape,
                      name="conv1"),
        layers.MaxPooling2D((2, 2), name="pool1"),

        # --- Block 2: Second convolutional layer ---
        layers.Conv2D(64, (3, 3), activation="relu", name="conv2"),
        layers.MaxPooling2D((2, 2), name="pool2"),

        # --- Classifier head ---
        layers.Flatten(name="flatten"),
        layers.Dense(128, activation="relu", name="dense1"),
        layers.Dropout(0.5, name="dropout"),
        layers.Dense(num_classes, activation="softmax", name="output"),
    ], name="digit_cnn")

    return model


# Build and compile the model
model = build_digit_cnn()

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## 5. Train the Model

In [ ]:
# Training hyperparameters
EPOCHS = 10
BATCH_SIZE = 128
VALIDATION_SPLIT = 0.1  # Use 10% of training data for validation

# Train the model
history = model.fit(
    X_train, y_train_cat,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    verbose=1,
)

print("\n✅ Training complete!")

## 6. Visualize Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Accuracy plot ---
ax1.plot(history.history["accuracy"], label="Train Accuracy", linewidth=2)
ax1.plot(history.history["val_accuracy"], label="Val Accuracy", linewidth=2)
ax1.set_title("Model Accuracy", fontsize=14, fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Loss plot ---
ax2.plot(history.history["loss"], label="Train Loss", linewidth=2)
ax2.plot(history.history["val_loss"], label="Val Loss", linewidth=2)
ax2.set_title("Model Loss", fontsize=14, fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Evaluate on Test Set

In [ ]:
# Evaluate on the held-out test set
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)

print(f"📊 Test Loss:     {test_loss:.4f}")
print(f"📊 Test Accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")

In [ ]:
# Detailed classification report
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print("\n📋 Classification Report:\n")
print(classification_report(y_test, y_pred_classes, target_names=[str(i) for i in range(10)]))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=range(10), yticklabels=range(10))
plt.title("Confusion Matrix", fontsize=16, fontweight="bold")
plt.xlabel("Predicted Digit", fontsize=12)
plt.ylabel("True Digit", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize some predictions
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle("Sample Predictions on Test Set", fontsize=16, fontweight="bold")

# Pick 10 random test samples
indices = np.random.choice(len(X_test), 10, replace=False)

for i, ax in enumerate(axes.flat):
    idx = indices[i]
    ax.imshow(X_test[idx].reshape(28, 28), cmap="gray")

    true_label = y_test[idx]
    pred_label = y_pred_classes[idx]
    confidence = y_pred[idx][pred_label] * 100

    color = "green" if true_label == pred_label else "red"
    ax.set_title(f"True: {true_label} | Pred: {pred_label}\n({confidence:.1f}%)",
                 fontsize=10, color=color)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 8. Save the Trained Model

We save the model in Keras `.keras` format so it can be loaded by the prediction script.

In [ ]:
# Create model directory
MODEL_DIR = "saved_model"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "digit_cnn.keras")

# Save the model
model.save(MODEL_PATH)
print(f"✅ Model saved to: {MODEL_PATH}")
print(f"   File size: {os.path.getsize(MODEL_PATH) / 1024:.1f} KB")

In [ ]:
# Verify the saved model loads correctly
loaded_model = keras.models.load_model(MODEL_PATH)
loaded_loss, loaded_acc = loaded_model.evaluate(X_test, y_test_cat, verbose=0)
print(f"✅ Loaded model test accuracy: {loaded_acc:.4f} (matches original: {abs(loaded_acc - test_accuracy) < 1e-6})")

## 9. Download the Model (Google Colab)

If you're running this in Google Colab, run the cell below to download the trained model file to your local machine.

In [ ]:
# Download model from Colab to local machine
try:
    from google.colab import files
    files.download(MODEL_PATH)
    print("📥 Download started!")
except ImportError:
    print("ℹ️  Not running in Colab. Model is saved locally at:", MODEL_PATH)

---

## ✅ Done!

**Next steps:**
1. Download the saved model (`saved_model/digit_cnn.keras`)
2. Place it in the `saved_model/` directory of the project
3. Use `predict.py` to predict digits from your own handwritten images

```bash
python predict.py path/to/your/digit_image.png
```